# 04 — Train: Extra Trees regression

Search a native multi-output Extra Trees for the configured station's direct multi-horizon forecast, then evaluate the selected candidate once on the sealed-test cohort.

**Inputs:** joined train/test feature artifacts and metadata contract  
**Outputs:** in-notebook metrics, an MLflow run hierarchy, and a model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Extra Trees is tree based and therefore uses the raw numeric predictors without scaling, same as Random Forest. Unlike Random Forest, its extra randomness comes from randomized split thresholds rather than bootstrap-sampled training rows. `EXTRA_TREES_N_ITER` sampled hyperparameter configurations are drawn once with the project seed and reused for every realized feature subset. Bootstrap sampling is controlled once for the whole notebook.

Any concrete values in this prose are illustrative examples, not run configuration. The executable constants in this notebook and imported values from `src/config.py` are authoritative for a run; Stage-3 feature metadata supplies the realized columns/subsets, and the saved manifest records the fitted model configuration. For example, the checked-in prose may describe 24 sampled configurations and disabled bootstrap, but the assignments in the code cell below determine what executes.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.model_selection import ParameterSampler  # type: ignore[import-untyped]
from tqdm.auto import tqdm

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    RANDOM_STATE,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.metrics import metric_tables
from src.plots import (
    cv_error_boxplots_figure,
    model_feature_subset_candidate_distribution_figure,
    predicted_vs_actual_figure,
    regime_aggregate_figure,
    regime_horizon_figure,
    test_error_boxplots_figure,
)
from src.regime_persistence import (
    regime_mlflow_metrics,
    regime_mlflow_params,
    sealed_test_regime_tables,
)
from src.training import (
    numeric_predictors,
    prediction_preview,
    summarize_cv_metrics,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
EXTRA_TREES_PARAM_DISTRIBUTIONS = {
    "n_estimators": [200, 400, 800],
    "max_depth": [8, 16, 32, None],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", 0.5, 1.0],
}
EXTRA_TREES_BOOTSTRAP = False
EXTRA_TREES_N_ITER = 24
EXTRA_TREES_N_JOBS = -1
MLFLOW_EXPERIMENT_NAME = "extra_trees"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"extra_trees_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"extra_trees_{station_id}.json"

## Shared model helpers

In [ ]:
from src.extra_trees import (
    build_extra_trees_estimator,
    normalize_candidate_key,
    save_extra_trees_manifest,
    select_candidate,
)

## Load the joined dataset

`load_joined_dataset` validates the feature contract, common eligibility cohort, ordered subsets, and expanding time-series folds before any model is fitted.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
FEATURE_SUBSETS = dataset.feature_subsets
train_rows = dataset.train_rows
test_rows = dataset.test_rows
INPUT_PARQUET_SHA256_PARAMS = dataset.input_hashes

## Joint subset and hyperparameter search

The sampled set is shared across subsets, resulting in `len(FEATURE_SUBSETS) × EXTRA_TREES_N_ITER` candidates and that count multiplied by `N_VALIDATION_FOLDS` fold fits. With six subsets, 24 sampled configurations, and five folds this would be 144 candidates and 720 fold fits; those numbers are examples only. The sealed test is not accessed until after deterministic CV selection.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splits = dataset.folds
validation_test_size = dataset.validation_test_size


sampled_hyperparameters = list(
    ParameterSampler(
        EXTRA_TREES_PARAM_DISTRIBUTIONS,
        n_iter=EXTRA_TREES_N_ITER,
        random_state=RANDOM_STATE,
    )
)
expected_candidate_keys = {
    normalize_candidate_key(
        subset=subset_name,
        max_depth=params["max_depth"],
        n_estimators=params["n_estimators"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
    )
    for subset_name in FEATURE_SUBSETS
    for params in sampled_hyperparameters
}
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
search_progress = tqdm(
    total=len(expected_candidate_keys) * N_VALIDATION_FOLDS,
    desc="Extra Trees CV search",
    unit="fit",
)
for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for params in sampled_hyperparameters:
        candidate_key = normalize_candidate_key(
            subset=subset_name,
            max_depth=params["max_depth"],
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
        )
        _, max_depth, n_estimators, min_samples_leaf, max_features = candidate_key
        label = f"md{max_depth or 'none'}_ne{n_estimators}_ml{min_samples_leaf}_mf{max_features}"
        fold_aggregate_rows = []
        fold_horizon_rows = []
        with mlflow.start_run(
            run_name=f"extra_trees_cv_{subset_name}_{label}",
            tags={
                "phase": "cv",
                "run_type": "candidate_parent",
                "subset": subset_name,
                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
            },
        ):
            mlflow.log_params(
                {
                    "phase": "cv",
                    "run_type": "candidate_parent",
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "feature_columns": json.dumps(feature_columns),
                    **INPUT_PARQUET_SHA256_PARAMS,
                    "max_depth": "None" if max_depth is None else str(max_depth),
                    "n_estimators": n_estimators,
                    "min_samples_leaf": min_samples_leaf,
                    "max_features": str(max_features),
                    "bootstrap": EXTRA_TREES_BOOTSTRAP,
                    "n_jobs": EXTRA_TREES_N_JOBS,
                    "random_state": RANDOM_STATE,
                    "n_validation_folds": N_VALIDATION_FOLDS,
                    "validation_test_size": validation_test_size,
                    "embargo_hours": EMBARGO_HOURS,
                    "selection_metric": CV_SELECTION_METRIC,
                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                    "common_train_rows": len(train_rows),
                }
            )
            for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
                cv_splits, start=1
            ):
                with mlflow.start_run(
                    run_name=f"extra_trees_cv_{subset_name}_{label}_fold_{fold_number}",
                    nested=True,
                    tags={
                        "phase": "cv",
                        "run_type": "fold",
                        "subset": subset_name,
                        "fold": str(fold_number),
                        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                    },
                ):
                    fold_train_rows = train_rows.iloc[fold_train_indices]
                    fold_validation_rows = train_rows.iloc[fold_validation_indices]
                    fold_model = build_extra_trees_estimator(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        max_features=max_features,
                        bootstrap=EXTRA_TREES_BOOTSTRAP,
                        n_jobs=EXTRA_TREES_N_JOBS,
                        random_state=RANDOM_STATE,
                    )
                    fold_model.fit(
                        numeric_predictors(fold_train_rows, feature_columns),
                        fold_train_rows[TARGET_COLUMNS],
                    )
                    fold_predictions = validate_predictions(
                        fold_model.predict(
                            numeric_predictors(fold_validation_rows, feature_columns)
                        ),
                        expected_rows=len(fold_validation_rows),
                        target_columns=TARGET_COLUMNS,
                        artifact_name="fold",
                    )
                    fold_aggregate, fold_per_horizon = metric_tables(
                        fold_validation_rows[TARGET_COLUMNS],
                        fold_predictions,
                        target_columns=TARGET_COLUMNS,
                        station_id=station_id,
                    )
                    fold_aggregate_rows.append(fold_aggregate.iloc[0])
                    fold_horizon_rows.append(fold_per_horizon)
                    search_progress.update(1)
                    mlflow.log_params(
                        {
                            "phase": "cv",
                            "run_type": "fold",
                            "subset": subset_name,
                            "fold": fold_number,
                            "n_estimators": n_estimators,
                            "max_depth": "None"
                            if max_depth is None
                            else str(max_depth),
                            "min_samples_leaf": min_samples_leaf,
                            "max_features": str(max_features),
                            "bootstrap": EXTRA_TREES_BOOTSTRAP,
                            "train_rows": len(fold_train_rows),
                            "validation_rows": len(fold_validation_rows),
                            "gap_rows": EMBARGO_HOURS,
                        }
                    )
                    mlflow.log_metrics(
                        {
                            "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                            "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                            "fold_me": float(fold_aggregate.iloc[0]["me"]),
                            "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                            **{
                                f"fold_mae_horizon_{row.horizon_hours:02d}": float(
                                    row.mae
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_rmse_horizon_{row.horizon_hours:02d}": float(
                                    row.rmse
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_me_horizon_{row.horizon_hours:02d}": float(
                                    row.me
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_r2_horizon_{row.horizon_hours:02d}": float(
                                    row.r2
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                        }
                    )
            parent_metrics = summarize_cv_metrics(
                pd.DataFrame(fold_aggregate_rows),
                pd.concat(fold_horizon_rows, ignore_index=True),
            )
            mlflow.log_metrics(
                {
                    metric_name: metric_value
                    for metric_name, metric_value in parent_metrics.items()
                }
            )
        cv_horizon_rows_by_candidate[candidate_key] = pd.concat(
            fold_horizon_rows, ignore_index=True
        )
        cv_results_rows.append(
            {
                "subset": subset_name,
                "feature_count": len(feature_columns),
                "max_depth": max_depth,
                "n_estimators": n_estimators,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
                "mae_mean": parent_metrics["cv_mae_mean"],
                "mae_std": parent_metrics["cv_mae_std"],
                "rmse_mean": parent_metrics["cv_rmse_mean"],
                "rmse_std": parent_metrics["cv_rmse_std"],
                "me_mean": parent_metrics["cv_me_mean"],
                "me_std": parent_metrics["cv_me_std"],
                "r2_mean": parent_metrics["cv_r2_mean"],
                "r2_std": parent_metrics["cv_r2_std"],
            }
        )
search_progress.close()
cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' and tags.phase = 'cv' and tags.run_type = 'candidate_parent'",
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' and tags.phase = 'cv' and tags.run_type = 'fold'",
)
if len(current_cv_runs) != len(expected_candidate_keys):
    raise ValueError(
        f"Current execution must produce {len(expected_candidate_keys)} candidate-parent runs, got {len(current_cv_runs)}"
    )
current_parent_keys = {
    normalize_candidate_key(
        subset=row["tags.subset"],
        max_depth=row["params.max_depth"],
        n_estimators=row["params.n_estimators"],
        min_samples_leaf=row["params.min_samples_leaf"],
        max_features=row["params.max_features"],
    )
    for _, row in current_cv_runs.iterrows()
}
if current_parent_keys != expected_candidate_keys:
    raise ValueError(
        "Current execution candidate-parent runs do not cover the sampled candidate set"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
current_fold_keys = {
    (
        *normalize_candidate_key(
            subset=row["tags.subset"],
            max_depth=row["params.max_depth"],
            n_estimators=row["params.n_estimators"],
            min_samples_leaf=row["params.min_samples_leaf"],
            max_features=row["params.max_features"],
        ),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (*candidate_key, fold_number)
    for candidate_key in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if current_fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every sampled candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
current_cv_result_keys = {
    normalize_candidate_key(
        subset=row.subset,
        max_depth=row.max_depth,
        n_estimators=row.n_estimators,
        min_samples_leaf=row.min_samples_leaf,
        max_features=row.max_features,
    )
    for row in cv_results.itertuples(index=False)
}
if current_cv_result_keys != expected_candidate_keys:
    raise ValueError("The in-memory CV result table does not match the candidate set")
(
    selected_subset,
    selected_max_depth,
    selected_n_estimators,
    selected_min_samples_leaf,
    selected_max_features,
) = select_candidate(cv_results, CV_SELECTION_METRIC)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
selected_candidate_key = (
    selected_subset,
    selected_max_depth,
    selected_n_estimators,
    selected_min_samples_leaf,
    selected_max_features,
)
selected_fold_horizon_metrics = cv_horizon_rows_by_candidate[selected_candidate_key]

## Retrain and score the selected candidate

Only the selected candidate is fitted on all eligible training rows before the single sealed-test pass.

In [ ]:
final_model = build_extra_trees_estimator(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    min_samples_leaf=selected_min_samples_leaf,
    max_features=selected_max_features,
    bootstrap=EXTRA_TREES_BOOTSTRAP,
    n_jobs=EXTRA_TREES_N_JOBS,
    random_state=RANDOM_STATE,
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns), train_rows[TARGET_COLUMNS]
)
test_predictions = validate_predictions(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns)),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
regime_definition, regime_aggregate_metrics, regime_horizon_metrics = (
    sealed_test_regime_tables(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        target_columns=TARGET_COLUMNS,
        station_id=station_id,
        quartile_cutoffs_cm=dataset.target_water_level_quartile_cutoffs_cm,
        quartile_reference_count=dataset.target_water_level_quartile_reference_count,
    )
)
if (
    not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all()
    or not np.isfinite(
        per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()
    ).all()
):
    raise ValueError("Extra Trees reported non-finite test metrics")
with mlflow.start_run(
    run_name=f"extra_trees_test_{selected_subset}",
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "subset": selected_subset,
            "feature_count": len(selected_feature_columns),
            "feature_columns": json.dumps(selected_feature_columns),
            **INPUT_PARQUET_SHA256_PARAMS,
            "max_depth": "None"
            if selected_max_depth is None
            else str(selected_max_depth),
            "n_estimators": selected_n_estimators,
            "min_samples_leaf": selected_min_samples_leaf,
            "max_features": str(selected_max_features),
            "bootstrap": EXTRA_TREES_BOOTSTRAP,
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            **regime_mlflow_params(regime_definition),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            **regime_mlflow_metrics(regime_aggregate_metrics, regime_horizon_metrics),
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_{metric}_horizon_{row.horizon_hours:02d}": float(
                    getattr(row, metric)
                )
                for row in per_horizon_metrics.itertuples()
                for metric in ("mae", "rmse", "me", "r2")
            },
        }
    )
    cv_figure = cv_error_boxplots_figure(
        selected_fold_horizon_metrics,
        TARGET_COLUMNS,
        title=f"Extra Trees CV errors — {selected_subset}",
    )
    mlflow.log_figure(cv_figure, "cv_rmse_mae_boxplots.png")
    plt.close(cv_figure)
    test_figure = test_error_boxplots_figure(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
        title=f"Extra Trees final-test errors — {selected_subset}",
    )
    mlflow.log_figure(test_figure, "test_error_boxplots.png")
    plt.close(test_figure)
    predicted_figure = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"Extra Trees predicted vs actual — {selected_subset}",
    )
    mlflow.log_figure(predicted_figure, "test_predicted_vs_actual.png")
    plt.close(predicted_figure)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
save_extra_trees_manifest(
    MODEL_METADATA_PATH,
    model_path=MODEL_PATH,
    execution_uuid=NOTEBOOK_EXECUTION_UUID,
    contract=contract,
    feature_subsets=FEATURE_SUBSETS,
    selected_subset=selected_subset,
    selected_max_depth=selected_max_depth,
    selected_n_estimators=selected_n_estimators,
    selected_min_samples_leaf=selected_min_samples_leaf,
    selected_max_features=selected_max_features,
    bootstrap=EXTRA_TREES_BOOTSTRAP,
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(test_rows, test_predictions, target_columns=TARGET_COLUMNS).head(
        PREDICTION_PREVIEW_ROWS
    )
)

# Extra Trees saved-model evaluation

This section validates the saved Extra Trees model manifest and presents the current execution's in-memory CV and sealed-test diagnostics. Results are stored in MLflow, not in the model manifest.


## Load the saved Extra Trees execution record

The joined dataset and model manifest are loaded here to validate the saved model against the current station, horizon, target columns, and selected subset columns. The comparison tables also use the current execution's in-memory `cv_results`, `aggregate_metrics`, and `per_horizon_metrics`; results are deliberately not read from the manifest.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.extra_trees import load_extra_trees_manifest, score_saved_model
from src.plots import forecast_window_figures

if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")

COMPARISON_PROCESSED_DIR = Path("data/processed/joined")
COMPARISON_METADATA_PATH = (
    COMPARISON_PROCESSED_DIR / "all_stations_feature_metadata.json"
)
COMPARISON_TRAIN_PATH = COMPARISON_PROCESSED_DIR / "all_stations_train_features.parquet"
COMPARISON_TEST_PATH = COMPARISON_PROCESSED_DIR / "all_stations_test_features.parquet"
COMPARISON_MODEL_PATH = Path("models") / f"extra_trees_{TARGET_STATION_ID}.joblib"
COMPARISON_MODEL_METADATA_PATH = (
    Path("models") / f"extra_trees_{TARGET_STATION_ID}.json"
)

comparison_dataset = load_joined_dataset(
    COMPARISON_METADATA_PATH,
    COMPARISON_TRAIN_PATH,
    COMPARISON_TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
comparison_contract = comparison_dataset.contract
COMPARISON_TARGET_COLUMNS = list(comparison_contract.target_columns)
comparison_test_rows = comparison_dataset.test_rows

extra_trees_manifest = load_extra_trees_manifest(
    COMPARISON_MODEL_METADATA_PATH,
    contract=comparison_contract,
    feature_subsets=comparison_dataset.feature_subsets,
)

## Inspect the recorded execution

The manifest is written once, after the sealed test has been scored, so an execution that crashed part-way leaves no record and the previous manifest survives untouched. The `execution_uuid` below is the same tag the MLflow runs of that execution carry, which is how the two views are tied together.


In [ ]:
selected_candidate_table = cv_results
selected_horizon_metrics = per_horizon_metrics.rename(
    columns={metric: f"test_{metric}" for metric in ("mae", "rmse", "me", "r2")}
)
candidate_depth = selected_candidate_table["max_depth"]
if extra_trees_manifest.selected_max_depth is None:
    max_depth_mask = candidate_depth.isna() | candidate_depth.eq("None")
else:
    max_depth_mask = pd.to_numeric(candidate_depth, errors="coerce").eq(
        extra_trees_manifest.selected_max_depth
    )
selected_candidate = selected_candidate_table.loc[
    selected_candidate_table["subset"].eq(extra_trees_manifest.selected_subset)
    & max_depth_mask
    & selected_candidate_table["n_estimators"].eq(
        extra_trees_manifest.selected_n_estimators
    )
    & selected_candidate_table["min_samples_leaf"].eq(
        extra_trees_manifest.selected_min_samples_leaf
    )
    & selected_candidate_table["max_features"].eq(
        extra_trees_manifest.selected_max_features
    )
].iloc[0]
selected_execution_summary = pd.DataFrame(
    [
        {
            "execution_uuid": extra_trees_manifest.execution_uuid,
            "manifest": str(COMPARISON_MODEL_METADATA_PATH),
            "candidate_count": len(selected_candidate_table),
            "selection_metric": CV_SELECTION_METRIC,
            "selected_subset": extra_trees_manifest.selected_subset,
            "selected_max_depth": extra_trees_manifest.selected_max_depth,
            "selected_n_estimators": extra_trees_manifest.selected_n_estimators,
            "selected_min_samples_leaf": extra_trees_manifest.selected_min_samples_leaf,
            "selected_max_features": extra_trees_manifest.selected_max_features,
            "bootstrap": extra_trees_manifest.bootstrap,
        }
    ]
)
display(selected_execution_summary)

## Compare cross-validation candidates

Candidate ranking uses only the recorded CV metrics and the configured selection metric; the sealed-test metrics are not used to rank candidates.


In [ ]:
candidate_columns = [
    "subset",
    "feature_count",
    "max_depth",
    "n_estimators",
    "min_samples_leaf",
    "max_features",
    "mae_mean",
    "mae_std",
    "rmse_mean",
    "rmse_std",
    "me_mean",
    "me_std",
    "r2_mean",
    "r2_std",
]
candidate_comparison_table = selected_candidate_table[candidate_columns].copy()
display(candidate_comparison_table)

### Compare performance across feature subsets

Each point is one hyperparameter candidate's aggregate CV mean. The boxes summarize the candidate distribution within each feature subset; they do not show fold-to-fold uncertainty.

In [ ]:
candidate_distribution_figure = model_feature_subset_candidate_distribution_figure(
    candidate_comparison_table,
    model_name="Extra Trees",
    hover_columns=(
        "feature_count",
        "max_depth",
        "n_estimators",
        "min_samples_leaf",
        "max_features",
    ),
)
display(candidate_distribution_figure)

## Visualize cross-validation error

The heatmap shows CV RMSE across feature subsets and max_depth values, faceted by n_estimators and averaged over the other sampled parameters. The line chart uses min_samples_leaf as the primary numeric axis and shows variation across the remaining sampled parameters. Unbounded depth is displayed as None and unsampled combinations remain gaps.


In [ ]:
plot_candidates = candidate_comparison_table.copy()
plot_candidates["max_depth_numeric"] = pd.to_numeric(
    plot_candidates["max_depth"], errors="coerce"
)
plot_candidates["max_depth_label"] = plot_candidates["max_depth_numeric"].map(
    lambda value: "None" if pd.isna(value) else str(int(value))
)
depth_labels = (
    plot_candidates[["max_depth_numeric", "max_depth_label"]]
    .drop_duplicates()
    .sort_values("max_depth_numeric", na_position="last")["max_depth_label"]
    .tolist()
)
n_estimators_facets = sorted(plot_candidates["n_estimators"].unique())
cv_rmse_heatmap_zmin = plot_candidates["rmse_mean"].min()
cv_rmse_heatmap_zmax = plot_candidates["rmse_mean"].max()
cv_rmse_heatmap_figure = make_subplots(
    rows=1,
    cols=len(n_estimators_facets),
    subplot_titles=[f"n_estimators={value}" for value in n_estimators_facets],
    shared_yaxes=True,
)
for column_index, n_estimators_value in enumerate(n_estimators_facets, start=1):
    facet_table = plot_candidates[
        plot_candidates["n_estimators"].eq(n_estimators_value)
    ]
    facet_heatmap_values = (
        facet_table.groupby(["subset", "max_depth_label"])["rmse_mean"]
        .mean()
        .unstack("max_depth_label")
        .reindex(columns=depth_labels)
        .sort_index(axis=0)
    )
    cv_rmse_heatmap_figure.add_trace(
        go.Heatmap(
            z=facet_heatmap_values.to_numpy(),
            x=depth_labels,
            y=facet_heatmap_values.index.tolist(),
            zmin=cv_rmse_heatmap_zmin,
            zmax=cv_rmse_heatmap_zmax,
            coloraxis="coloraxis",
            hovertemplate="Subset=%{y}<br>max_depth=%{x}<br>CV RMSE=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column_index,
    )
cv_rmse_heatmap_figure.update_layout(
    title="Extra Trees candidate CV RMSE by feature subset, max_depth, and n_estimators (mean over other sampled hyperparameters)",
    coloraxis={"colorscale": "Viridis", "colorbar": {"title": "CV RMSE"}},
)
cv_rmse_heatmap_figure.update_xaxes(title_text="max_depth")
cv_rmse_heatmap_figure.update_yaxes(title_text="Feature subset", col=1)
display(cv_rmse_heatmap_figure)

In [ ]:
cv_rmse_by_min_samples_leaf_figure = go.Figure()
for subset_name, subset_candidates in candidate_comparison_table.groupby(
    "subset", sort=True
):
    subset_summary = (
        subset_candidates.groupby("min_samples_leaf")["rmse_mean"]
        .agg(["mean", "std"])
        .sort_index()
    )
    cv_rmse_by_min_samples_leaf_figure.add_trace(
        go.Scatter(
            x=subset_summary.index,
            y=subset_summary["mean"],
            mode="lines+markers",
            name=subset_name,
            error_y={
                "type": "data",
                "array": subset_summary["std"].fillna(0.0),
                "visible": True,
            },
        )
    )
cv_rmse_by_min_samples_leaf_figure.update_layout(
    title="Extra Trees CV RMSE versus min_samples_leaf, by feature subset (mean over other sampled hyperparameters)",
    xaxis_title="min_samples_leaf",
    yaxis_title="CV RMSE",
)
display(cv_rmse_by_min_samples_leaf_figure)

## Inspect selected-candidate sealed-test performance

These values belong only to the candidate recorded by the selected sealed-test run. The final chart shows its MAE and RMSE at each available forecast horizon.


In [ ]:
selected_candidate_sealed_test_summary = pd.DataFrame(
    [
        {
            "execution_uuid": extra_trees_manifest.execution_uuid,
            "subset": selected_candidate["subset"],
            "feature_count": selected_candidate["feature_count"],
            "max_depth": selected_candidate["max_depth"],
            "n_estimators": selected_candidate["n_estimators"],
            "min_samples_leaf": selected_candidate["min_samples_leaf"],
            "max_features": selected_candidate["max_features"],
            "cv_mae_mean": selected_candidate["mae_mean"],
            "cv_rmse_mean": selected_candidate["rmse_mean"],
            "cv_me_mean": selected_candidate["me_mean"],
            "cv_r2_mean": selected_candidate["r2_mean"],
            **{
                f"test_{metric}": float(aggregate_metrics.iloc[0][metric])
                for metric in ("mae", "rmse", "me", "r2")
            },
        }
    ]
)
display(selected_candidate_sealed_test_summary)
sealed_test_horizon_figure = go.Figure(
    [
        go.Scatter(
            x=selected_horizon_metrics["horizon_hours"],
            y=selected_horizon_metrics[metric_name],
            mode="lines+markers",
            name=metric_name.upper(),
        )
        for metric_name in ("test_mae", "test_rmse", "test_me", "test_r2")
    ]
)
sealed_test_horizon_figure.update_layout(
    title="Selected Extra Trees candidate sealed-test error by horizon",
    xaxis_title="Forecast horizon (hours)",
    yaxis_title="Error",
)
display(sealed_test_horizon_figure)

## Reload and score the saved Extra Trees model

Reloads the saved Extra Trees model and scores the eligible sealed-test cohort on the manifest's selected feature columns, without retraining or changing the stored prediction semantics.


In [ ]:
comparison_prediction_values = score_saved_model(
    extra_trees_manifest,
    COMPARISON_MODEL_PATH,
    comparison_test_rows,
)
print(
    f"Scored {len(comparison_test_rows):,} eligible sealed-test rows with "
    f"{len(COMPARISON_TARGET_COLUMNS)} horizons using the saved Extra Trees model."
)

In [ ]:
comparison_prediction_columns = [
    f"prediction_{target_column}" for target_column in COMPARISON_TARGET_COLUMNS
]
comparison_prediction_table = (
    comparison_test_rows[["timestamp", *COMPARISON_TARGET_COLUMNS]]
    .reset_index(drop=True)
    .rename(columns={"timestamp": "issue_time"})
)
comparison_prediction_table = pd.concat(
    [
        comparison_prediction_table,
        pd.DataFrame(
            comparison_prediction_values,
            columns=comparison_prediction_columns,
        ),
    ],
    axis=1,
)
comparison_prediction_table["issue_time"] = pd.to_datetime(
    comparison_prediction_table["issue_time"], utc=True
)

comparison_issue_times = comparison_prediction_table["issue_time"]
comparison_horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
comparison_horizon_labels = [f"H+{horizon:02d}" for horizon in comparison_horizons]
comparison_time_series_frames = []
for horizon in comparison_horizons:
    target_column = COMPARISON_TARGET_COLUMNS[horizon - 1]
    prediction_column = comparison_prediction_columns[horizon - 1]
    valid_times = comparison_issue_times + pd.to_timedelta(horizon, unit="h")
    comparison_time_series_frames.append(
        go.Frame(
            name=comparison_horizon_labels[horizon - 1],
            data=[
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[target_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Actual",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Actual=%{y:.3f}<extra></extra>",
                ),
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[prediction_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Prediction",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Prediction=%{y:.3f}<extra></extra>",
                ),
            ],
        )
    )
comparison_time_series_steps = [
    {
        "label": comparison_horizon_labels[horizon - 1],
        "method": "animate",
        "args": [[comparison_horizon_labels[horizon - 1]], {"mode": "immediate"}],
    }
    for horizon in comparison_horizons
]
comparison_time_series_figure = go.Figure(
    data=comparison_time_series_frames[0].data,
    frames=comparison_time_series_frames,
    layout={
        "title": "Saved Extra Trees predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": comparison_time_series_steps,
            }
        ],
    },
)
extra_trees_prediction_time_series_figure = comparison_time_series_figure
display(comparison_time_series_figure)

## Inspect best and worst Extra Trees forecast windows

The following plots use the saved-model predictions and select sealed-test issue times by the RMSE calculated across every configured forecast horizon. A context window is eligible only when the target-station water-level series contains every hourly observation across the notebook's configured context window, with no imputed observations.


In [ ]:
extra_trees_forecast_window_figures = forecast_window_figures(
    comparison_prediction_table,
    comparison_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=comparison_prediction_columns,
    target_columns=COMPARISON_TARGET_COLUMNS,
    horizons=comparison_horizons,
    label_prefix="Extra Trees",
)

### Best


In [ ]:
best_extra_trees_forecast_window_figure = extra_trees_forecast_window_figures["best"]
display(best_extra_trees_forecast_window_figure)

### Worst


In [ ]:
worst_extra_trees_forecast_window_figure = extra_trees_forecast_window_figures["worst"]
display(worst_extra_trees_forecast_window_figure)

## Compare absolute and signed errors

Each box contains all eligible sealed-test errors for one horizon. Absolute-error markers reuse the current execution's per-horizon MAE/RMSE values; signed errors follow the convention `prediction - actual`.


In [ ]:
comparison_actual_values = comparison_test_rows[COMPARISON_TARGET_COLUMNS].to_numpy(
    dtype=float
)
signed_errors = comparison_prediction_values - comparison_actual_values
absolute_errors = np.abs(signed_errors)

absolute_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_mae"],
            mode="markers",
            name="MAE",
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_rmse"],
            mode="markers",
            name="RMSE",
        ),
    ]
)
absolute_error_boxplot_figure.update_layout(
    title="Saved Extra Trees absolute errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Absolute error",
)

display(absolute_error_boxplot_figure)

In [ ]:
signed_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=signed_errors.mean(axis=0),
            mode="markers",
            name="Mean error",
        ),
    ]
)
signed_error_boxplot_figure.update_layout(
    title="Saved Extra Trees signed errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Signed error (prediction - actual)",
)
signed_error_boxplot_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)
display(signed_error_boxplot_figure)

## Inspect the selected Extra Trees model

The saved estimator is a native multi-output `ExtraTreesRegressor`. This section plots impurity-based feature importances for every selected input feature and reports the selected hyperparameters together with the fitted tree, feature, and output counts.

Impurity importance is the normalized total reduction in split impurity attributed to a feature across the forest. It is useful for describing this fitted model, but it is not causal evidence: correlated predictors can split importance between them, high-cardinality or noisy continuous features can be favored, and the values can be unstable across samples. Permutation importance or held-out ablations are better for assessing predictive contribution.


In [ ]:
from IPython.display import Markdown
from joblib import load as load_joblib

importance_model = load_joblib(COMPARISON_MODEL_PATH)
selected_feature_columns = list(extra_trees_manifest.selected_feature_columns)
importance_scores = np.asarray(importance_model.feature_importances_, dtype=float)
if len(importance_scores) != len(selected_feature_columns):
    raise ValueError(
        "Saved Extra Trees feature importance length does not match the manifest"
    )
importance_table = (
    pd.DataFrame({"feature": selected_feature_columns, "importance": importance_scores})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance_figure = go.Figure(
    go.Bar(
        x=importance_table["importance"], y=importance_table["feature"], orientation="h"
    )
)
feature_importance_figure.update_layout(
    title="Saved Extra Trees impurity-based feature importance",
    xaxis_title="Mean decrease in impurity",
    yaxis_title="Feature",
    yaxis={"categoryorder": "total ascending"},
    height=max(400, 20 * len(importance_table)),
)
display(feature_importance_figure)
architecture_summary_table = pd.DataFrame(
    [
        {
            "selected_subset": extra_trees_manifest.selected_subset,
            "selected_max_depth": extra_trees_manifest.selected_max_depth,
            "selected_n_estimators": extra_trees_manifest.selected_n_estimators,
            "selected_min_samples_leaf": extra_trees_manifest.selected_min_samples_leaf,
            "selected_max_features": extra_trees_manifest.selected_max_features,
            "bootstrap": extra_trees_manifest.bootstrap,
            "random_state": extra_trees_manifest.random_state,
            "fitted_tree_count": len(importance_model.estimators_),
            "fitted_feature_count": int(importance_model.n_features_in_),
            "fitted_output_count": int(importance_model.n_outputs_),
        }
    ]
)
display(
    Markdown(
        f"The saved model uses subset **{extra_trees_manifest.selected_subset}**, maximum depth **{extra_trees_manifest.selected_max_depth}**, **{extra_trees_manifest.selected_n_estimators}** estimators, minimum leaf size **{extra_trees_manifest.selected_min_samples_leaf}**, and max features **{extra_trees_manifest.selected_max_features}**."
    )
)
display(architecture_summary_table)

## Q1–Q4 aggregate diagnostics


In [ ]:
display(
    regime_aggregate_figure(
        regime_aggregate_metrics, regime_group="quartile", model_label="Extra Trees"
    )
)

## Q1–Q4 MAE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="quartile",
        metric="mae",
        model_label="Extra Trees",
    )
)

## Q1–Q4 RMSE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="quartile",
        metric="rmse",
        model_label="Extra Trees",
    )
)

## Q1–Q4 ME by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="quartile",
        metric="me",
        model_label="Extra Trees",
    )
)

## Alarm aggregate diagnostics


In [ ]:
display(
    regime_aggregate_figure(
        regime_aggregate_metrics, regime_group="alarm", model_label="Extra Trees"
    )
)

## Alarm MAE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="alarm",
        metric="mae",
        model_label="Extra Trees",
    )
)

## Alarm RMSE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="alarm",
        metric="rmse",
        model_label="Extra Trees",
    )
)

## Alarm ME by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="alarm",
        metric="me",
        model_label="Extra Trees",
    )
)